<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_51_opentelemetry_for_llms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🔭 **Module 51 — OpenTelemetry for LLMs (traces, metrics, logs — one standard)** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)


# Module 51 — OpenTelemetry for LLMs

> M50 covered metrics. **OpenTelemetry (OTel)** unifies the other two pillars — **traces** and **logs** — into one vendor-neutral SDK + protocol. For LLM apps it's transformative: a single trace shows you the full path of one user request across **gateway → retriever → reranker → LLM call → tool call**, with token counts, prompts, and latencies on each step.
>
> OTel has become the de-facto standard. Datadog, Honeycomb, Jaeger, Tempo, Splunk, New Relic, Langfuse — they all speak OTel. Instrument once, send anywhere.

### What you'll cover
1. The trace mental model — span, parent, context propagation
2. The OTel architecture — SDK · Collector · backend
3. Setup — install OTel + run the Collector
4. **Instrument a Python app** — manual spans
5. Auto-instrumentation for FastAPI / requests / SQLAlchemy
6. **GenAI semantic conventions** — the standard span attributes for LLM calls
7. **OpenLLMetry / Traceloop / Langfuse / Phoenix** — the LLM ecosystem on top of OTel
8. **Sampling** — head, tail, parent-based; what to actually use
9. Exporting to a backend — Jaeger / Tempo / Datadog / Langfuse
10. Putting it together — RAG trace, debug a real bug


## 1 · Trace mental model

A **trace** is a tree of **spans**. Each span has a name, start/end times, status, and attributes. A span has a **parent** (its caller), and the root span is the request itself.

```
Trace: chat_request_42
└─ span: gateway.handle              [320 ms]
   └─ span: retriever.query           [80 ms]
   │   └─ span: vector_db.search      [70 ms]
   ├─ span: reranker.score            [40 ms]
   └─ span: llm.completion            [180 ms]
       ├─ tool_call: web_search        [80 ms]
       └─ tool_call: db.lookup         [60 ms]
```

**Context propagation** is what stitches spans across processes. When `gateway` calls `retriever` over gRPC, the trace ID and span ID travel in HTTP headers (`traceparent`, `tracestate`). The retriever's spans land in the same trace.

You read traces by asking questions:
- *Which step took the time?* (a single panel reveals the answer)
- *Where do errors come from?* (red spans)
- *What was the prompt + the response?* (LLM-specific span attributes)


## 2 · OTel architecture

```
┌─ your app (Python, Go, Java, …) ─┐         ┌── OTel Collector ──┐         ┌─ backend ─┐
│  OTel SDK ──── OTLP/gRPC ───────►├──HTTP──►│ receivers/processors├──────►│ Jaeger    │
│  (spans, metrics, logs)          │         │ /exporters          │       │ Tempo     │
└──────────────────────────────────┘         └─────────────────────┘       │ Datadog   │
                                                                            │ Langfuse  │
                                                                            └───────────┘
```

| Component | Role |
|---|---|
| **OTel API** | what your code calls (`tracer.start_as_current_span(...)`) |
| **OTel SDK** | implementation: samplers, processors, exporters |
| **OTLP** | the wire protocol (gRPC / HTTP+protobuf) |
| **Collector** | sidecar/standalone process: receive → process → fan-out |
| **Backend** | where you actually look at the data |

**Always run a Collector** between apps and backends. Why?
- Apps batch + ship over OTLP only. Backend-specific stuff (sampling, redaction, multi-vendor fan-out) lives in the Collector.
- Switching backends becomes a Collector config change, not an app redeploy.

## 3 · Setup

In [ ]:
!pip -q install \
    opentelemetry-api \
    opentelemetry-sdk \
    opentelemetry-exporter-otlp \
    opentelemetry-instrumentation-requests \
    opentelemetry-instrumentation-fastapi

Run the Collector locally — one container:
```bash
# minimal collector that prints traces to stdout
docker run -p 4317:4317 -p 4318:4318 \
    -v $PWD/otel-config.yaml:/etc/otel/config.yaml \
    otel/opentelemetry-collector-contrib \
    --config=/etc/otel/config.yaml
```

Where `otel-config.yaml` is:
```yaml
receivers:
  otlp: { protocols: { grpc: {}, http: {} } }
processors:
  batch: {}
exporters:
  debug: { verbosity: detailed }
service:
  pipelines:
    traces: { receivers: [otlp], processors: [batch], exporters: [debug] }
```


## 4 · Manual spans — the core API

In [ ]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor, ConsoleSpanExporter
from opentelemetry.sdk.resources import Resource

# Wire up: provider → processor → exporter
provider = TracerProvider(resource=Resource.create({"service.name": "rag-app"}))
provider.add_span_processor(BatchSpanProcessor(ConsoleSpanExporter()))   # for demo; OTLP in prod
trace.set_tracer_provider(provider)

tracer = trace.get_tracer(__name__)

In [ ]:
import time, random

def retrieve(query):
    with tracer.start_as_current_span("retriever.query") as span:
        span.set_attribute("query.length", len(query))
        time.sleep(0.05)
        results = ["doc-1", "doc-2", "doc-3"]
        span.set_attribute("retriever.k", len(results))
        return results

def call_llm(query, ctx):
    with tracer.start_as_current_span("llm.completion") as span:
        span.set_attribute("llm.model", "qwen2.5-0.5b")
        span.set_attribute("llm.prompt.tokens", random.randint(40, 80))
        time.sleep(0.15)
        out = f"answer to {query}"
        span.set_attribute("llm.completion.tokens", len(out.split()))
        return out

def chat(query):
    with tracer.start_as_current_span("chat") as parent:
        parent.set_attribute("user.id", "u123")
        ctx = retrieve(query)
        out = call_llm(query, ctx)
        parent.set_attribute("response.length", len(out))
        return out

chat("what is RAG?")

**Span hygiene.**
- Use **verbs in span names**: `retriever.query`, `llm.completion`. Not `retriever_called`.
- Set **status** on errors: `span.set_status(Status(StatusCode.ERROR, "msg"))`.
- Record exceptions: `span.record_exception(e)`.
- Don't blow cardinality on attributes (no full prompts as attribute keys; use **events** for big payloads).

## 5 · Auto-instrumentation

In [ ]:
from opentelemetry.instrumentation.requests import RequestsInstrumentor
from opentelemetry.instrumentation.fastapi import FastAPIInstrumentor

# one line per library — every requests.get() / FastAPI route
# emits spans automatically with HTTP method, status, URL etc.
RequestsInstrumentor().instrument()
# FastAPIInstrumentor.instrument_app(app)
print("auto-instrumentation enabled for: requests, fastapi")

There are **120+ official auto-instrumentations** — `psycopg`, `redis`, `aio_pika`, `boto3`, `sqlalchemy`, `grpcio`, `httpx`, `starlette`, `aiokafka`, …  Run:

```bash
opentelemetry-bootstrap -a install   # detects installed libs and adds matching otel packages
opentelemetry-instrument python app.py   # zero-code instrumentation
```

For most CRUD apps that's enough — you skip §4 entirely.

## 6 · GenAI semantic conventions

OTel's **semantic conventions** standardise span attribute names so dashboards work across vendors. The **GenAI** spec (stable since 2024) covers LLM calls:

| Attribute | Example |
|---|---|
| `gen_ai.system` | `"openai"`, `"anthropic"`, `"vllm"`, `"ollama"` |
| `gen_ai.request.model` | `"gpt-4o-mini"` |
| `gen_ai.request.temperature` | `0.7` |
| `gen_ai.request.top_p` | `0.9` |
| `gen_ai.usage.input_tokens` | `512` |
| `gen_ai.usage.output_tokens` | `128` |
| `gen_ai.response.finish_reasons` | `["stop"]` |
| `gen_ai.operation.name` | `"chat"`, `"embedding"`, `"text_completion"` |
| Event: `gen_ai.user.message` | `{ "content": "…", "role": "user" }` |
| Event: `gen_ai.assistant.message` | `{ "content": "…", "role": "assistant" }` |

Use **events** (not attributes) for the actual prompt/response text — they have higher cardinality budgets and avoid blowing up attribute storage.

## 7 · The LLM ecosystem on top of OTel

| Project | What it adds |
|---|---|
| **OpenLLMetry / Traceloop** | one-line wrapping of OpenAI / Anthropic / LangChain / LlamaIndex SDKs to emit GenAI-spec spans automatically |
| **Langfuse** | OTel-compatible LLM observability backend with prompt versioning, eval, datasets, cost tracking |
| **Arize Phoenix** | open-source LLM observability + evals; great for offline analysis |
| **LangSmith** | LangChain's hosted tracing (proprietary protocol; OTel adapter exists) |
| **Datadog / New Relic / Honeycomb** | the generic backends; all accept OTLP and have LLM dashboards |

```python
# OpenLLMetry style — one line and every OpenAI call becomes a properly-attributed span
from traceloop.sdk import Traceloop
Traceloop.init(app_name="rag-app")
# ... your normal OpenAI/Anthropic/LangChain/Llamaindex code unchanged ...
```

This is the modern default for LLM apps: don't write GenAI spans by hand, let `traceloop-sdk` (or similar) do it, then look at them in **Langfuse** or **Phoenix**.

## 8 · Sampling

You can't (and shouldn't) trace 100% of traffic at scale.

| Strategy | When |
|---|---|
| **AlwaysOn / AlwaysOff** | dev / never |
| **TraceIdRatioBased(0.01)** | 1% of all traces — cheap, lossy |
| **ParentBased** | sample child if parent was sampled (preserves full traces) — almost always wrap one of the others |
| **Tail-based** (Collector) | decide *after* the trace finishes: keep it if errors / slow / contains LLM call |

```yaml
# tail-based sampling in the Collector
processors:
  tail_sampling:
    decision_wait: 10s
    policies:
      - { name: errors,    type: status_code,   status_code: { status_codes: [ERROR] } }
      - { name: slow,      type: latency,       latency: { threshold_ms: 2000 } }
      - { name: every-llm, type: string_attribute, string_attribute: { key: gen_ai.system, values: [openai, anthropic, vllm] } }
      - { name: random,    type: probabilistic, probabilistic: { sampling_percentage: 5 } }
```

The point: **keep the interesting traces; drop the boring ones.**

## 9 · Exporters — OTLP everywhere

In [ ]:
# swap ConsoleSpanExporter for OTLP — same code, different destination
from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter

otlp = OTLPSpanExporter(endpoint="http://localhost:4317", insecure=True)
provider.add_span_processor(BatchSpanProcessor(otlp))
print("now exporting to OTLP collector at :4317")

Common Collector destinations:
```yaml
exporters:
  otlp/jaeger:    { endpoint: jaeger:4317,         tls: { insecure: true } }
  otlp/tempo:     { endpoint: tempo:4317 }
  otlphttp/honeycomb: { endpoint: https://api.honeycomb.io, headers: { x-honeycomb-team: $${HONEYCOMB_KEY} } }
  otlp/langfuse:  { endpoint: https://us.cloud.langfuse.com:443, headers: { authorization: "Basic $${LF_AUTH}" } }
  datadog:        { api: { key: $${DD_API_KEY}, site: datadoghq.com } }
```

You can fan-out: send the same trace to **two** backends (e.g. Jaeger for dev + Datadog for prod alerts).

## 10 · Putting it together — debugging a real bug

A user reports "the chatbot was super slow on Tuesday at 3 PM."

1. Open the trace explorer, filter to that user's traces in that window.
2. Sort by duration desc — top trace was 8.2 s.
3. Open it. The waterfall shows:
   - `chat` 8.2 s → mostly `retriever.query` 7.9 s
   - inside `retriever.query`: `vector_db.search` 7.8 s
4. Inspect `vector_db.search` attributes: `db.system="qdrant"`, `db.collection="prod_v3"`, `db.k=20`.
5. Cross-reference with metrics (M50): Qdrant CPU was pegged that minute.
6. Confirmed cause: a missed `payload_index` on a frequent filter → ANN walked the whole index.

**You couldn't have done that with metrics alone.** Metrics tell you *something* was slow. Traces tell you *which span* and *why*.

### Anti-patterns
- ❌ Tracing 100% of traffic at scale → cost explosion. Use sampling.
- ❌ Putting full prompts as **attributes** instead of **events** → cardinality bomb.
- ❌ Logging tokens / PII in spans without redaction → compliance nightmare. Add a Collector **redaction processor**.
- ❌ Manual spans inside auto-instrumented frameworks → duplicate spans. Pick one.
- ❌ Different `service.name` per replica → trace explorer becomes useless. Use the **same** service name across replicas; differ on `service.instance.id`.


## ✅ Recap
- Trace = tree of spans, glued by **context propagation**.
- Architecture: SDK → OTLP → Collector → backend(s). Always run a Collector.
- **Manual spans** for your own logic; **auto-instrumentation** for libs.
- **GenAI semantic conventions** + **OpenLLMetry / Langfuse** make LLM tracing one-line.
- **Tail-based sampling** in the Collector keeps the interesting traces and drops the rest.
- Traces answer "where did the time go?" — metrics (M50) only tell you that something is slow.

Next: **M52 — Go for AI Backends** (the language most ML platforms are written in).
